<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/Cost_of_Zendo_1_(CoZ1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cost of Sleep System (CoSS) Schema

This schema is designed to be fractally scalable, allowing for nested sub-systems and components. Each item will have a `name`, `type`, `quantity`, `unit_cost`, `total_cost`, and an optional `components` list for further breakdown.

In [1]:
co_ss = {
    'name': 'Cost of Sleep System (CoSS)',
    'type': 'system',
    'components': [
        {
            'name': 'Bedding Sub-system (BS)',
            'type': 'sub_system',
            'components': [
                {'name': 'mattress', 'quantity': 1, 'unit_cost': None, 'total_cost': None},
                {'name': 'pillows', 'quantity': 4, 'unit_cost': None, 'total_cost': None},
                {'name': 'bed frame', 'quantity': 1, 'unit_cost': None, 'total_cost': None},
                {'name': 'mattress protector (waterproof)', 'quantity': 3, 'unit_cost': None, 'total_cost': None},
                {'name': 'bedsheets', 'quantity': 4, 'unit_cost': None, 'total_cost': None},
                {'name': 'pillow cases', 'quantity': 4, 'unit_cost': None, 'total_cost': None}
            ]
        }
        # Other sub-systems will go here, e.g., 'Lighting Sub-system', 'Sound Sub-system'
    ]
}

# Display the initial schema
import json
print(json.dumps(co_ss, indent=2))

{
  "name": "Cost of Sleep System (CoSS)",
  "type": "system",
  "components": [
    {
      "name": "Bedding Sub-system (BS)",
      "type": "sub_system",
      "components": [
        {
          "name": "mattress",
          "quantity": 1,
          "unit_cost": null,
          "total_cost": null
        },
        {
          "name": "pillows",
          "quantity": 4,
          "unit_cost": null,
          "total_cost": null
        },
        {
          "name": "bed frame",
          "quantity": 1,
          "unit_cost": null,
          "total_cost": null
        },
        {
          "name": "mattress protector (waterproof)",
          "quantity": 3,
          "unit_cost": null,
          "total_cost": null
        },
        {
          "name": "bedsheets",
          "quantity": 4,
          "unit_cost": null,
          "total_cost": null
        },
        {
          "name": "pillow cases",
          "quantity": 4,
          "unit_cost": null,
          "total_cost": null
 

### Explanation of the Schema:

*   **`name`**: The human-readable name of the system, sub-system, or component.
*   **`type`**: Categorizes the item (e.g., `system`, `sub_system`, `component`). This can be extended as needed.
*   **`quantity`**: The number of units for a given component.
*   **`unit_cost`**: Placeholder for the cost of a single unit. Initially set to `None`.
*   **`total_cost`**: Placeholder for the total cost (`quantity * unit_cost`). Initially set to `None`.
*   **`components`**: A list of nested dictionaries, representing items or sub-systems that are part of the current item. This is the key to the fractal scaling.

This structure allows you to easily add more sub-systems (e.g., 'Ambient Lighting Sub-system', 'Sound Management Sub-system') under the main `co_ss` 'components' list, and further break down any sub-system into its individual components. You can then write functions to traverse this structure and calculate total costs at any level.

## Generate Google Sheet from CoSS Schema

To generate and populate a Google Sheet, we'll use the `gspread` library for Google Sheets interaction and `PyDrive` for Google Drive management (like creating folders). We'll first install these libraries and then set up authentication.

In [2]:
%%capture
!pip install gspread pydrive

In [3]:
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import gspread
import pandas as pd
import json

# Authenticate with Google account
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
gc = gspread.authorize(GoogleCredentials.get_application_default())

print("Google authentication complete.")

TypeError: Credentials need to be from either oauth2client or from google-auth.

In [4]:
def find_or_create_folder(folder_name, parent_id=None):
    """Finds a folder by name, or creates it if it doesn't exist."""
    query = f"'{{parent_id}}' in parents and mimeType='application/vnd.google-apps.folder' and title='{folder_name}' and trashed=false"
    if parent_id is None:
        # Search in root if no parent_id is specified
        query = f"'root' in parents and mimeType='application/vnd.google-apps.folder' and title='{folder_name}' and trashed=false"
    else:
        query = f"'{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and title='{folder_name}' and trashed=false"

    file_list = drive.ListFile({'q': query}).GetList()
    if file_list:
        return file_list[0]['id']
    else:
        print(f"Creating folder: {folder_name}")
        folder_metadata = {
            'title': folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
        }
        if parent_id:
            folder_metadata['parents'] = [{'id': parent_id}]
        folder = drive.CreateFile(folder_metadata)
        folder.Upload()
        return folder['id']

def create_folder_structure(path):
    """Creates a nested folder structure if it doesn't exist."""
    parts = path.split('/')
    parent_id = None
    # Start from 'My Drive' root if specified
    if parts[0].lower() == 'my drive':
        # 'root' is typically 'My Drive' for the authenticated user
        parent_id = 'root'
        parts = parts[1:] # Remove 'My Drive' from parts

    for part in parts:
        if part:
            parent_id = find_or_create_folder(part, parent_id)
    return parent_id

# Define the target directory path
target_directory_path = 'My Drive/2026/zendo/SSOT/z1'

# Create the folder structure and get the final folder ID
target_folder_id = create_folder_structure(target_directory_path)

if target_folder_id:
    print(f"Target folder ID: {target_folder_id}")
else:
    print("Could not find or create the target folder.")

InvalidConfigError: Invalid client secrets file ('Error opening file', 'client_secrets.json', 'No such file or directory', 2)

In [5]:
def flatten_schema(schema_dict):
    """Flattens the nested CoSS schema into a list of dictionaries suitable for a DataFrame."""
    flat_data = []

    def process_components(components, parent_name=None):
        for component in components:
            current_data = {
                'System': schema_dict['name'] if parent_name is None else '',
                'Sub-system': component['name'] if component['type'] == 'sub_system' else '',
                'Component': component['name'] if component['type'] != 'sub_system' else '',
                'Quantity': component.get('quantity'),
                'Unit Cost': component.get('unit_cost'),
                'Total Cost': component.get('total_cost')
            }
            flat_data.append(current_data)

            if 'components' in component and component['components']:
                process_components(component['components'], component['name'])

    process_components(schema_dict['components'])
    return flat_data

# Flatten the `co_ss` schema
flat_co_ss_data = flatten_schema(co_ss)

# Convert to a pandas DataFrame
df_coss = pd.DataFrame(flat_co_ss_data)
df_coss.set_index(['System', 'Sub-system', 'Component'], inplace=True)
df_coss = df_coss.loc[~df_coss.index.get_level_values('Component').duplicated(keep='first')]

# Reset index to make 'System', 'Sub-system', 'Component' columns again
df_coss.reset_index(inplace=True)

# Adjust 'System' and 'Sub-system' columns to only show the value once for readability
for col in ['System', 'Sub-system']:
    last_val = None
    for i in range(len(df_coss)):
        current_val = df_coss.loc[i, col]
        if current_val == last_val:
            df_coss.loc[i, col] = ''
        last_val = current_val

display(df_coss)

KeyError: 'type'

In [6]:
# Create a new Google Sheet
sheet_name = 'Zendo Z1 CoSS Bedding Sub-system'
try:
    spreadsheet = gc.create(sheet_name, folder_id=target_folder_id)
    spreadsheet.share('', role='writer', type='anyone') # Share with anyone to easily view/edit
    print(f"Spreadsheet '{sheet_name}' created at: {spreadsheet.url}")

    # Select the first worksheet
    worksheet = spreadsheet.worksheet('Sheet1')

    # Update the worksheet with DataFrame data
    # Convert DataFrame to a list of lists, including headers
    worksheet.update([df_coss.columns.values.tolist()] + df_coss.values.tolist())

    print("Data successfully written to Google Sheet.")
except gspread.exceptions.DuplicateSpreadsheet as e:
    print(f"A spreadsheet with the name '{sheet_name}' already exists. Please delete it or choose a new name. Error: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

AttributeError: module 'gspread.exceptions' has no attribute 'DuplicateSpreadsheet'